# Preprocessing Pipeline
## ArtEmis Dataset - Image & Label Preparation

In [5]:
# Import
import torch
import sys
sys.path.append('..')

from src.preprocessing import load_data, get_splits, get_class_weights
from src.dataset import ArtEmisDataset, get_transforms, get_dataloaders

In [6]:
df, le = load_data("../data/dataset_final.csv")
train_df, val_df, test_df = get_splits(df, subset_size=3000)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"\nEmotion: Label mapping:")
for emotion, label in zip(le.classes_, range(len(le.classes_))):
    print(f"  {emotion}: {label}")

Train: 2400 | Val: 300 | Test: 300

Emotion: Label mapping:
  amusement: 0
  anger: 1
  awe: 2
  contentment: 3
  disgust: 4
  excitement: 5
  fear: 6
  sadness: 7


In [7]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
class_weights = get_class_weights(train_df, device)

print("Class weights:")
for emotion, weight in zip(le.classes_, class_weights):
    print(f"  {emotion}: {weight:.4f}")

Class weights:
  amusement: 7.1429
  anger: 1.8519
  awe: 3.5714
  contentment: 1.4019
  disgust: 0.8475
  excitement: 4.4776
  fear: 0.5199
  sadness: 0.3333


In [8]:
train_transforms, val_test_transforms = get_transforms()
print("Train transforms:", train_transforms)
print("\nVal/Test transforms:", val_test_transforms)

Train transforms: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomRotation(degrees=[-15.0, 15.0], interpolation=nearest, expand=False, fill=0)
    ColorJitter(brightness=(0.7, 1.3), contrast=(0.7, 1.3), saturation=(0.7, 1.3), hue=None)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

Val/Test transforms: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


In [9]:
IMAGE_DIR = "../data/images"
train_loader, val_loader, test_loader = get_dataloaders(train_df, val_df, test_df, IMAGE_DIR)

images, labels = next(iter(train_loader))
print(f"Batch image shape: {images.shape}")
print(f"Batch label shape: {labels.shape}")

Batch image shape: torch.Size([32, 3, 224, 224])
Batch label shape: torch.Size([32])


## Preprocessing Summary
- Labels encoded: 8 emotion classes (0-7)
- Split: 39,244 train / 4,906 val / 4,906 test (80/10/10, stratified)
- Images resized to 224×224 and normalised (ImageNet mean/std)
- Train augmentation: random flip, rotation, colour jitter
- Class weights computed to handle severe imbalance
- DataLoaders verified: batch shape [32, 3, 224, 224]